# 音階テスト & 波形テスト (WAV)

音を **WAV ファイル**として生成し、FFT で「狙った音程・波形どおりに鳴っているか」を検証するノートブックです。

- 音の出力手段: WAV (16bit PCM, `wave` 標準ライブラリ)
- 平均律 A4 = 440 Hz
- 各セルは上から順に実行してください。音はセル内で再生 (`IPython.display.Audio`)、波形・スペクトルはインライン描画します。

**依存**: `numpy`(必須), `matplotlib`(描画), `IPython`(再生)。`scipy` は不要。


## 0. 準備 (import と既定パラメータ)

`IPython` / `pandas` が無い環境でも動くよう、再生は WAV 保存、表は print にフォールバックします。

In [2]:
import math
import wave
from dataclasses import dataclass, field

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# --- IPython (音の再生) は任意 ---
try:
    from IPython.display import Audio, display
    HAVE_IPY = True
except Exception:
    HAVE_IPY = False
    def display(*a, **k):     # フォールバック
        for x in a:
            print(x)

# --- pandas (結果表) は任意 ---
try:
    import pandas as pd
    HAVE_PD = True
except Exception:
    HAVE_PD = False

# ---- 既定パラメータ ----
SAMPLE_RATE = 44100      # サンプリング周波数 [Hz]
A4_FREQ     = 440.0      # 基準音 A4 [Hz]
DEFAULT_AMP = 0.8        # 既定振幅 (フルスケール=1.0)
TOLERANCE_CENTS = 10.0   # 合否しきい値 (±10 cent)

def show_table(rows, title=None):
    """結果リストを pandas DataFrame（無ければ整形print）で表示。"""
    if title:
        print(title)
    if HAVE_PD:
        df = pd.DataFrame(rows); display(df); return df
    if rows:
        cols = list(rows[0])
        print(" | ".join(cols))
        for r in rows:
            print(" | ".join(str(r[c]) for c in cols))
    return rows

def play(samples, sr=SAMPLE_RATE, filename="_play.wav"):
    """IPython があれば再生ウィジェット、無ければ WAV 保存。"""
    if HAVE_IPY:
        display(Audio(np.asarray(samples), rate=sr))
    else:
        write_wav(filename, samples, sr)
        print(f"(IPython無し) {filename} を保存しました")

print("準備完了: SR =", SAMPLE_RATE, "Hz,  A4 =", A4_FREQ, "Hz",
      "| IPython:", HAVE_IPY, "| pandas:", HAVE_PD)

ModuleNotFoundError: No module named 'matplotlib'

## 1. 音名 ↔ 周波数 (平均律)

`A4 = 440 Hz`、半音 = 2^(1/12) 倍。MIDI ノート番号を介して相互変換します。

In [ ]:
_NOTE_BASE = {"C":0,"D":2,"E":4,"F":5,"G":7,"A":9,"B":11}
_ACC = {"#":1,"\u266f":1,"b":-1,"\u266d":-1,"":0}
_SEMI = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]

def note_to_midi(note: str) -> int:
    """'A4','C#5','Bb3' -> MIDI 番号 (A4=69)。"""
    note = note.strip()
    letter = note[0].upper()
    if letter not in _NOTE_BASE:
        raise ValueError(f"不正な音名: {note!r}")
    i = 1; acc = ""
    if i < len(note) and note[i] in _ACC:
        acc = note[i]; i += 1
    octave = int(note[i:])
    return (octave + 1) * 12 + _NOTE_BASE[letter] + _ACC[acc]

def midi_to_freq(m: float) -> float:
    return A4_FREQ * (2.0 ** ((m - 69) / 12.0))

def note_to_freq(note: str) -> float:
    return midi_to_freq(note_to_midi(note))

def freq_to_note(freq: float):
    """周波数 -> (最も近い音名, セント誤差)。"""
    if freq <= 0:
        return ("---", 0.0)
    mf = 69 + 12 * math.log2(freq / A4_FREQ)
    m = int(round(mf))
    return (_SEMI[m % 12] + str(m // 12 - 1), (mf - m) * 100.0)

# 動作確認
for n in ["A4", "C4", "C#5", "Bb3"]:
    print(f"{n:>4} -> {note_to_freq(n):8.3f} Hz   逆変換 {freq_to_note(note_to_freq(n))}")

## 2. 波形生成 (sine / square / sawtooth / triangle)

- `harmonics = 0`: 理想波形（高域にエイリアスを含む）
- `harmonics > 0`: 倍音の加算合成（バンドリミット）でエイリアスを抑制

In [ ]:
def _phase(freq, dur, sr=SAMPLE_RATE):
    n = int(round(dur * sr))
    return 2*np.pi*freq*(np.arange(n)/sr)

def sine_wave(freq, dur, sr=SAMPLE_RATE):
    return np.sin(_phase(freq, dur, sr))

def square_wave(freq, dur, sr=SAMPLE_RATE, harmonics=0):
    if harmonics <= 0:
        return np.sign(np.sin(_phase(freq, dur, sr)))
    ph = _phase(freq, dur, sr); out = np.zeros_like(ph); k = 1
    while k <= harmonics and freq*k < sr/2:
        out += np.sin(k*ph)/k; k += 2
    return (4/np.pi)*out

def sawtooth_wave(freq, dur, sr=SAMPLE_RATE, harmonics=0):
    if harmonics <= 0:
        t = np.arange(int(round(dur*sr)))*freq/sr
        return 2*(t - np.floor(t+0.5))
    ph = _phase(freq, dur, sr); out = np.zeros_like(ph); k = 1
    while k <= harmonics and freq*k < sr/2:
        out += ((-1)**(k+1))*np.sin(k*ph)/k; k += 1
    return (2/np.pi)*out

def triangle_wave(freq, dur, sr=SAMPLE_RATE, harmonics=0):
    if harmonics <= 0:
        t = np.arange(int(round(dur*sr)))*freq/sr
        return 2*np.abs(2*(t - np.floor(t+0.5))) - 1
    ph = _phase(freq, dur, sr); out = np.zeros_like(ph); k = 1; sign = 1.0
    while k <= harmonics and freq*k < sr/2:
        out += sign*np.sin(k*ph)/(k*k); sign = -sign; k += 2
    return (8/np.pi**2)*out

WAVEFORMS = {"sine":sine_wave, "square":square_wave,
             "sawtooth":sawtooth_wave, "triangle":triangle_wave}
print("波形:", list(WAVEFORMS))

## 3. ADSR エンベロープ & 1音レンダリング

In [ ]:
@dataclass
class ADSR:
    attack: float = 0.01
    decay: float = 0.05
    sustain: float = 0.7
    release: float = 0.08
    def apply(self, x, sr=SAMPLE_RATE):
        n = len(x); env = np.ones(n)
        a = min(int(self.attack*sr), n)
        d = min(int(self.decay*sr), max(n-a, 0))
        r = min(int(self.release*sr), max(n-a-d, 0))
        s = max(n-a-d-r, 0); i = 0
        if a: env[i:i+a] = np.linspace(0,1,a,endpoint=False); i += a
        if d: env[i:i+d] = np.linspace(1,self.sustain,d,endpoint=False); i += d
        if s: env[i:i+s] = self.sustain; i += s
        if r: env[i:i+r] = np.linspace(self.sustain,0,r,endpoint=True)
        return x * env

def render_note(freq, dur, waveform="sine", amp=DEFAULT_AMP,
                sr=SAMPLE_RATE, envelope=None, harmonics=0):
    gen = WAVEFORMS[waveform]
    w = gen(freq, dur, sr) if waveform == "sine" else gen(freq, dur, sr, harmonics=harmonics)
    w = w * amp
    return envelope.apply(w, sr) if envelope is not None else w

def render_silence(dur, sr=SAMPLE_RATE):
    return np.zeros(int(round(dur*sr)))

## 4. WAV 入出力 (16bit PCM, モノラル)

In [ ]:
def write_wav(path, samples, sr=SAMPLE_RATE):
    pcm = (np.clip(samples, -1, 1) * 32767).astype("<i2")
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())

def read_wav(path):
    with wave.open(path, "rb") as wf:
        sr = wf.getframerate(); nch = wf.getnchannels()
        raw = wf.readframes(wf.getnframes())
    data = np.frombuffer(raw, dtype="<i2").astype(np.float64)/32768.0
    if nch > 1:
        data = data.reshape(-1, nch).mean(axis=1)
    return data, sr

# テスト: 書いて読み戻す
write_wav("_tmp_test.wav", render_note(note_to_freq("A4"), 0.5), SAMPLE_RATE)
_d, _sr = read_wav("_tmp_test.wav")
print("read_wav:", len(_d), "samples @", _sr, "Hz")

## 5. FFT 解析 (ピッチ検出 / スペクトル / THD)

放物線補間でサブビン精度のピッチを推定し、THD(全高調波歪み率)も求めます。

In [ ]:
def detect_pitch(x, sr=SAMPLE_RATE):
    x = np.asarray(x, float)
    if x.size == 0 or np.max(np.abs(x)) < 1e-9:
        return 0.0
    spec = np.abs(np.fft.rfft(x*np.hanning(x.size)))
    if spec.size < 3:
        return 0.0
    k = int(np.argmax(spec[1:]) + 1)
    if 1 <= k < spec.size-1:
        a,b,c = (math.log(spec[k-1]+1e-12), math.log(spec[k]+1e-12), math.log(spec[k+1]+1e-12))
        den = a - 2*b + c
        delta = 0.5*(a-c)/den if abs(den) > 1e-12 else 0.0
    else:
        delta = 0.0
    return (k+delta)*sr/x.size

def spectrum(x, sr=SAMPLE_RATE):
    mag = np.abs(np.fft.rfft(x*np.hanning(len(x))))
    freqs = np.fft.rfftfreq(len(x), 1/sr)
    peak = mag.max() if mag.size and mag.max() > 0 else 1.0
    return freqs, mag/peak

def thd_percent(x, f0, sr=SAMPLE_RATE, n=10):
    mag = np.abs(np.fft.rfft(x*np.hanning(len(x))))
    freqs = np.fft.rfftfreq(len(x), 1/sr)
    def amp(f):
        sel = (freqs >= f*0.97) & (freqs <= f*1.03)
        return float(mag[sel].max()) if np.any(sel) else 0.0
    v1 = amp(f0)
    if v1 <= 0:
        return 0.0
    return 100*math.sqrt(sum(amp(f0*k)**2 for k in range(2, n+1)))/v1

def rms(x):
    x = np.asarray(x, float)
    return float(np.sqrt(np.mean(x**2))) if x.size else 0.0

def cents_error(detected, target):
    return 1200*math.log2(detected/target) if detected > 0 else float("nan")

---
# 6. 音階テスト

音階の各音を WAV 生成 → 読み戻して FFT 検証 → 音名どおりの周波数かをセント単位で判定します。

In [ ]:
SCALE_PATTERNS = {
    "chromatic":  [0,1,2,3,4,5,6,7,8,9,10,11,12],
    "major":      [0,2,4,5,7,9,11,12],
    "minor":      [0,2,3,5,7,8,10,12],
    "pentatonic": [0,2,4,7,9,12],
}

def run_scale_test(scale="major", root="C4", waveform="sine",
                   harmonics=0, bpm=120):
    sr = SAMPLE_RATE; beat = 60/bpm
    env = ADSR(0.01, 0.04, 0.75, 0.06)
    root_m = note_to_midi(root)
    rows = []; parts = []
    for semi in SCALE_PATTERNS[scale]:
        name, _ = freq_to_note(midi_to_freq(root_m+semi))
        target = note_to_freq(name)
        x = render_note(target, beat, waveform, DEFAULT_AMP, sr, env, harmonics)
        write_wav("_tmp.wav", x, sr)
        d, _ = read_wav("_tmp.wav")
        det = detect_pitch(d, sr); ce = cents_error(det, target)
        rows.append({"音名":name, "目標Hz":round(target,2), "検出Hz":round(det,2),
                     "誤差cent":round(ce,2),
                     "判定":"PASS" if abs(ce) <= TOLERANCE_CENTS else "FAIL"})
        parts.append(x)
    seq = np.concatenate(parts)
    show_table(rows)
    passed = sum(1 for r in rows if r["判定"] == "PASS")
    print(f"結果: {passed}/{len(rows)} PASS  (±{TOLERANCE_CENTS:.0f} cent)")
    return seq, sr

seq, sr = run_scale_test("major", "C4", "sine")
play(seq, sr)  # ▶ 再生 (IPython無しなら _play.wav 保存)

## 6.1 カエルの歌 (ドレミファミレド…)

メロディを (音名, 長さ[拍]) で定義して鳴らします。

In [ ]:
FROG = [("C4",1),("D4",1),("E4",1),("F4",1),("E4",1),("D4",1),("C4",1),("REST",1),
        ("E4",1),("F4",1),("G4",1),("A4",1),("G4",1),("F4",1),("E4",1),("REST",1),
        ("C4",1),("REST",1),("C4",1),("REST",1),("C4",1),("REST",1),("C4",1),("REST",1),
        ("C4",.5),("C4",.5),("D4",.5),("D4",.5),("E4",.5),("E4",.5),("F4",.5),("F4",.5),
        ("E4",1),("D4",1),("C4",1)]

def play_melody(score, waveform="sine", bpm=120, harmonics=0):
    sr = SAMPLE_RATE; beat = 60/bpm
    env = ADSR(0.01, 0.04, 0.75, 0.06); parts = []
    for note, length in score:
        if note == "REST":
            parts.append(render_silence(length*beat, sr))
        else:
            parts.append(render_note(note_to_freq(note), length*beat,
                                     waveform, DEFAULT_AMP, sr, env, harmonics))
    return np.concatenate(parts), sr

frog, sr = play_melody(FROG, waveform="triangle", bpm=132)
write_wav("frog.wav", frog, sr)
print("frog.wav を保存しました")
play(frog, sr, "frog.wav")  # ▶ 再生

---
# 7. 波形テスト

同じ音 (既定 A4) で 4 波形を生成し、時間波形・スペクトルを描画、ピッチ/RMS/THD を解析します。

In [ ]:
def run_waveform_test(note="A4", harmonics=0, dur=1.0):
    sr = SAMPLE_RATE; f0 = note_to_freq(note)
    names = list(WAVEFORMS)
    fig, axes = plt.subplots(len(names), 2, figsize=(11, 2.2*len(names)))
    rows = []
    for ax_row, wf in zip(axes, names):
        x = render_note(f0, dur, wf, DEFAULT_AMP, sr, None, harmonics)
        write_wav(f"{note}_{wf}.wav", x, sr)
        d, _ = read_wav(f"{note}_{wf}.wav")
        det = detect_pitch(d, sr)
        rows.append({"波形":wf, "検出Hz":round(det,2),
                     "誤差cent":round(cents_error(det,f0),2),
                     "RMS":round(rms(d),3), "Peak":round(float(np.max(np.abs(d))),3),
                     "THD%":round(thd_percent(d,f0,sr),2),
                     "判定":"PASS" if abs(cents_error(det,f0)) <= TOLERANCE_CENTS else "FAIL"})
        # 時間波形 (3周期)
        n_show = min(len(d), int(round(3*sr/f0)))
        ax_row[0].plot(np.arange(n_show)/sr*1000, d[:n_show])
        ax_row[0].set_title(f"{wf} (time)"); ax_row[0].set_xlabel("ms"); ax_row[0].grid(alpha=.3)
        # スペクトル
        fr, mg = spectrum(d, sr); sel = fr <= f0*12
        ax_row[1].plot(fr[sel], mg[sel], color="#d62728")
        ax_row[1].set_title(f"{wf} (spectrum)"); ax_row[1].set_xlabel("Hz"); ax_row[1].grid(alpha=.3)
    fig.tight_layout(); plt.show()
    show_table(rows)
    passed = sum(1 for r in rows if r["判定"] == "PASS")
    print(f"結果: {passed}/{len(rows)} PASS  (±{TOLERANCE_CENTS:.0f} cent)")
    return rows

rows = run_waveform_test("A4", harmonics=25)

## 7.1 各波形を聴き比べる

In [ ]:
f0 = note_to_freq("A4")
for wf in WAVEFORMS:
    print(wf)
    x = render_note(f0, 1.0, wf, DEFAULT_AMP, SAMPLE_RATE, ADSR(), harmonics=25)
    play(x, SAMPLE_RATE, f"A4_{wf}.wav")

---
## まとめ

- **音階テスト** (§6): 各音を WAV 生成→FFT 検証し、音名どおりの周波数かを ±10 cent で自動判定。`run_scale_test("minor","A3","square",harmonics=15)` のように引数変更可。
- **波形テスト** (§7): 4 波形の時間波形・スペクトル・THD を比較。理想波形は `harmonics=0`、バンドリミットは `harmonics>0`。
- すべて WAV (16bit PCM) ベース。`.py` 版 (`scale_test.py` / `waveform_test.py` / `tonelib.py`) と同じロジックです。
